# 03 · Full refit (tmQM + LnQM) — 2026-10 statistics

Deployed DB: `data/refit/final/constraints_2026-10-17.json` (+ sidecar); new fits write separately. One unattended run from the cleaned source data to `data/refit/final/constraints.json` (+ `.meta.json`).
Every step is a function in `refit/run_refit.py` (also runnable from a terminal:
`python refit/run_refit.py all --workers 2`); this notebook calls the same functions and shows their tables.

**What this refit changes** (decided 09-28 – 10-01; `claude/refit-spec-additions-2026-09-30.md`, prerefit checklist):

| item | change |
|---|---|
| A1 | every mixture is evidence-weighted: each complex counts once per key (1/m per row); a component needs ≥ 10 effective complexes; K by BIC on the effective count (the old `n // 30` cap forced keys with < 60 rows to one component) |
| A2 | **Q1 and Q2 partial pooling**: keys with ≥ 50 complexes are fitted on their own data, every other key is MAP-adapted from its ladder parent (relevance r chosen here on held-out complexes) |
| A3 | Q2 components are Student-t (ν ∈ {2, 4, 10} by likelihood) — insurance against catastrophic scores |
| A4 | Q2 key carries the **site-pair class** (ideal angle of the two sites in the best-fit assignment: c90, c180, c120, …); kept at every back-off level; haptic faces are their own family; chelate pairs kept |
| A5 | inert Q2 padding (scorer) |
| A6 | Q5 well model: weighted VMM, one component per well, minority off-well tail dropped; rows **folded by rotor symmetry** at extraction (scorer folds automatically for this DB) |
| B1 | η7 / η8 faces get their own labels (`C_eta7`, `C_eta8`) instead of polluting the η2 table |
| B3 | Q5 aryl rotors: ortho-substituent count in the key (`free_o0/1/2`, back-off `no_ortho`) |
| Q6 | contact-**onset** wall (μ = 0.1 % quantile, σ from the 1 %–0.1 % spread) and polar-H keys (`Hp` = H on N/O/F facing an acceptor), so real contacts and H-bonds are no longer penalised |
| A8 | S_orient incl. the per-substituent haptic rows, QG / QP priors, benchmark hold-out (N48 screen + RING10) — all in this one run |

**Time** (12 workers): cleaning ~10 min, extraction ~1–2 h, selection ~1 h, fit ~1 h.
Outputs are cached: re-running skips a stage whose outputs exist (set `FORCE = True` to redo).

In [ ]:
import sys, os, json, time
from pathlib import Path
import pandas as pd, numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jacobik').exists())
sys.path.insert(0, str(ROOT / 'refit'))
import run_refit as RR

cfg = RR.Config(
    root=str(ROOT),
    tm_dir='data/tmQM',     # tmQM: xyz/ bo/ tmQM_y.csv (splits X1-X3)
    ln_dir='data/refit/source/lnQM',     # LnQM exporter output (Angstrom); missing -> tmQM only
    ln_xyz_in_bohr=False,
    workers=2,
    q1_variants=('mo_state', 'cn_state', 'm_sub_early', 'orig', 'no_state', 'metal_first'),
    pool_r=(4.0, 10.0, 25.0),
)
FORCE = False          # True: recompute every stage below
INSTALL = False        # True: copy the new DB into src/jacobik/ at the end (backup kept)
cfg.force = FORCE
cfg.setup()
import refit_lib as R
pd.set_option('display.width', 200)
print(ROOT, '| refit dir', cfg.refit, '| benchmark codes held out:', len(R.benchmark_codes()))

## 1. Oxidation-state label audit (fast, no geometry)
Reject reasons and the per-metal OS histogram; a chemically odd state is a SMILES artefact and only feeds its own thin key and the OS-free levels.

In [ ]:
AUD = RR.stage_audit(cfg)
pd.DataFrame(AUD['histogram']).fillna(0).astype(int).T.head(40)

## 2. Contact cleaning + sharding
Thresholds are derived from the data (`reports/contacts/contact_filter_params.json`); dropped contacts are kept with their reason.

In [ ]:
THR = RR.stage_clean(cfg)
print('through-bond angle cut (deg):', THR.get('theta'))
dr = [pd.read_parquet(p) for p in sorted(cfg.contacts.glob('dropped_contacts_*.parquet'))]
pd.concat(dr).groupby('reason').size() if dr else 'no dropped-contact files'

## 3. Extraction (Q1–Q6, S_orient rows, QG/QP/QB) + merge, benchmark complexes removed

In [ ]:
ST = RR.stage_extract(cfg)
ds = [d for d in cfg.datasets if (cfg.obs / f'observations_Q1_{d}.parquet').exists()]
rows = {d: {q: len(pd.read_parquet(cfg.obs / f'observations_{q}_{d}.parquet', columns=['csd_code'])) for q in RR.QS} for d in ds}
print(pd.DataFrame(rows).T)
leak = {d: int(pd.read_parquet(cfg.obs / f'observations_Q1_{d}.parquet', columns=['csd_code']).csd_code.isin(R.benchmark_codes()).sum()) for d in ds}
print('benchmark rows left (must be 0):', leak); assert not any(leak.values())
if ST: print(json.dumps({d: {k: v for k, v in s.items() if k != 'rows'} for d, s in ST.items()}, indent=1))

## 4. Sanity of the new keys (before fitting)

In [ ]:
import ast
q2 = pd.concat([pd.read_parquet(cfg.obs / f'observations_Q2_{d}.parquet') for d in ds])
k2 = q2.key.map(ast.literal_eval)
q2['pc'] = k2.map(lambda k: k[1]); q2['geom'] = k2.map(lambda k: k[0])
print('Q2 rows per (geometry, pair class): median angle (deg)')
print(q2.groupby(['geom', 'pc']).value.agg(['size', 'median']).query('size > 500').round(1).to_string())
q1 = pd.concat([pd.read_parquet(cfg.obs / f'observations_Q1_{d}.parquet') for d in ds])
sub = q1.key.map(lambda k: ast.literal_eval(k)[3])
print('\nhaptic Q1 (centroid distance, A):'); print(q1[sub.str.startswith(('C_eta', 'C_haptic', 'C_allyl'))].groupby(sub).value.describe()[['count', '50%', 'min', 'max']].round(2))
q5 = pd.concat([pd.read_parquet(cfg.obs / f'observations_Q5_{d}.parquet', columns=['key']) for d in ds])
print('\nQ5 contexts:', q5.key.map(lambda k: ast.literal_eval(k)[4]).value_counts().to_dict())
q6 = pd.concat([pd.read_parquet(cfg.obs / f'observations_Q6_{d}.parquet', columns=['key', 'value']) for d in ds])
print('\nQ6 polar-H keys:'); print(q6[q6.key.str.contains('Hp')].groupby('key').value.describe()[['count', 'min', '50%']].round(2).head(15))

## 5. Held-out selection (complexes split by CSD-code hash, 1 in 10 held out)
Q1: ladder variant × relevance factor r; Q2: r. Every held-out row is resolved exactly as the runtime does. The pick is
the best mean held-out log p per complex (a state-keeping Q1 variant within 0.01 is preferred). `sparse` = held-out
complexes whose exact key had < 10 training complexes (where pooling matters). Also the S_orient hold-out.

In [ ]:
SEL = RR.stage_select(cfg)
print(json.dumps({k: v for k, v in SEL.items() if k != 'QO'}, indent=1, default=str))
pd.read_csv(cfg.hold / 'pool_Q1.csv').sort_values('mean', ascending=False).round(3)

In [ ]:
pd.read_csv(cfg.hold / 'pool_Q2.csv').round(3)

In [ ]:
print('S_orient held-out log p / dim:', json.dumps(SEL['QO'].get('loglik'), indent=1, default=float))
pd.read_csv(cfg.hold / 'qo_holdout_by_class.csv', index_col=0).head(25).round(3)

## 6. Full fit (all data): Q1/Q2 pooled with the selected settings, Q3–Q6, S_orient, priors, sidecar

In [ ]:
FIT = RR.stage_fit(cfg)
print(json.dumps({k: v for k, v in (FIT or {}).items() if k != 'Q1-Q6'}, indent=1, default=str))
R.db_summary(cfg.final / 'constraints.json')

## 7. Checks
Far Q1 components (second-sphere contamination that survived cleaning: should be empty or explainable), the Q6 walls, pooled shares.

In [ ]:
CHK = RR.stage_check(cfg)
FAR = R.far_component_audit(cfg.final / 'constraints.json', far_A=3.0); print(len(FAR), 'far components'); FAR.head(25)

In [ ]:
new = json.loads((cfg.final / 'constraints.json').read_text())
q6 = pd.DataFrame({k[3:]: v for k, v in new.items() if k.startswith('Q6:')}).T[['mu', 'sigma', 'n']].astype(float)
q6.loc[[k for k in ["('C', 'C')", "('C', 'H')", "('H', 'H')", "('H', 'O')", "('Hp', 'O')", "('Hp', 'N')", "('N', 'O')", "('O', 'O')"] if k in q6.index]].round(3)

## 8. DOE aqua validation (optional; validation only, never fitted on)
Needs `data/10.25584 - 3019314/*/opt-xyz` and, for the baseline, the old `src/` tree with its DB. Separate processes so
`src` and `src` never share an interpreter.

In [ ]:
RUN_DOE = False
import glob, subprocess
def doe(src, db, tag, extra=()):
    out = cfg.refit / f'doe_{tag}.json'
    d = next(iter(glob.glob(str(ROOT / 'data' / '10.25584 - 3019314' / '*' / 'opt-xyz'))))
    subprocess.run([sys.executable, str(ROOT / 'refit' / 'doe_aqua.py'), '--src', str(src), '--db', str(db), '--out', str(out), '--doe-dir', d, *extra], check=True, cwd=str(ROOT))
    return json.loads(out.read_text())
def summ(res):
    r = [x for x in res['rows'] if not x['system'].startswith('Fe')]; e = np.array([x['got'] - x['ref'] for x in r])
    th = np.array([x['tilt_got'] - x['tilt_ref'] for x in r])
    return dict(n=len(r), MAE=np.abs(e).mean(), bias=e.mean(), tiltMAE_deg=np.nanmean(np.abs(th)))
if RUN_DOE:
    DOE = {'refit': doe(ROOT / 'src', cfg.final / 'constraints.json', 'refit')}
    if (ROOT / 'src' / 'jacobik' / 'constraints.json').exists():
        DOE['baseline(src)'] = doe(ROOT / 'src', ROOT / 'src' / 'jacobik' / 'constraints.json', 'baseline')
    print(pd.DataFrame({k: summ(v) for k, v in DOE.items()}).T.round(3))

## 9. Install (gated)
Copies the DB and its sidecar into `src/jacobik/` (backup `constraints.pre_refit.json`). Benchmarks can use the DB in place instead (`JACOBIK_DB`, notebook 01).

In [ ]:
cfg.install = INSTALL
RR.stage_install(cfg)